In [ ]:
from pathlib import Path
import shutil
import subprocess
import sys
import zipfile

INPUT_ROOT = Path("/kaggle/input")
WORK_ROOT = Path("/kaggle/working")
PROJECT_WORK = WORK_ROOT / "cbs-project-src"
DATA_ROOT = WORK_ROOT / "cbs-data"

def is_project(folder):
    return (
        (folder / "pyproject.toml").is_file()
        and (folder / "src" / "cosmic_black_swan" / "cli.py").is_file()
    )

# Use a project folder from the attached input if one exists.
project_roots = [
    p.parent for p in INPUT_ROOT.rglob("pyproject.toml")
    if is_project(p.parent)
]

if project_roots:
    if len(project_roots) != 1:
        raise RuntimeError(f"Found multiple project folders: {project_roots}")
    shutil.copytree(project_roots[0], PROJECT_WORK, dirs_exist_ok=True)
    project_roots = [PROJECT_WORK]
else:
    # Otherwise find the ZIP containing the project source.
    matching_zips = []
    for archive_path in INPUT_ROOT.rglob("*.zip"):
        try:
            with zipfile.ZipFile(archive_path) as archive:
                names = [n.replace("\\", "/") for n in archive.namelist()]
                has_project = any(n.endswith("pyproject.toml") for n in names)
                has_cli = any(n.endswith("src/cosmic_black_swan/cli.py") for n in names)
                if has_project and has_cli:
                    matching_zips.append(archive_path)
        except zipfile.BadZipFile:
            pass

    if len(matching_zips) != 1:
        raise RuntimeError(
            "Expected one project ZIP in /kaggle/input; found: "
            + str(matching_zips)
        )

    PROJECT_WORK.mkdir(parents=True, exist_ok=True)
    root = PROJECT_WORK.resolve()

    with zipfile.ZipFile(matching_zips[0]) as archive:
        for member in archive.infolist():
            relative = Path(member.filename.replace("\\", "/"))
            if relative.is_absolute() or ".." in relative.parts:
                raise RuntimeError(f"Unsafe ZIP path: {member.filename}")
            destination = (root / relative).resolve()
            if destination != root and root not in destination.parents:
                raise RuntimeError(f"Unsafe ZIP path: {member.filename}")
            archive.extract(member, root)

    project_roots = [
        p.parent for p in PROJECT_WORK.rglob("pyproject.toml")
        if is_project(p.parent)
    ]

if len(project_roots) != 1:
    raise RuntimeError(f"Could not identify one project source folder: {project_roots}")

PROJECT_ROOT = project_roots[0]
DATA_ROOT.mkdir(parents=True, exist_ok=True)

print("Project source:", PROJECT_ROOT)
print("Data output:", DATA_ROOT)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-e", str(PROJECT_ROOT)],
    check=True,
)

In [ ]:
fetch_file = PROJECT_ROOT / "src" / "cosmic_black_swan" / "fetch.py"
fetch_source = fetch_file.read_text(encoding="utf-8")

if "for page_number in range(1, 101)" not in fetch_source:
    start = fetch_source.find("        observations = Observations.query_criteria(")
    end = fetch_source.find('        if "productFilename" not in frame', start)

    if start < 0 or end < 0:
        raise RuntimeError("Could not find the expected MAST query section in fetch.py")

    paginated_query = """        product_pages = []
        for page_number in range(1, 11):
            print(f"Sector {sector}: querying MAST page {page_number}", flush=True)
            observations = Observations.query_criteria(
                obs_collection="TESS",
                dataproduct_type="timeseries",
                sequence_number=int(sector),
                pagesize=100,
                page=page_number,
            )
            if observations is None or len(observations) == 0:
                break

            page_products = Observations.get_product_list(observations)
            if page_products is None or len(page_products) == 0:
                break
            product_pages.append(page_products)

            if len(observations) < 100:
                break

        if not product_pages:
            continue

        from astropy.table import vstack
        products = vstack(product_pages, metadata_conflicts="silent")
        frame = _mast_table_frame(products)
"""

    fetch_source = fetch_source[:start] + paginated_query + fetch_source[end:]

batch_loop = """    for offset in range(0, len(pending_rows), batch_size):
        batch = pending_rows[offset : offset + batch_size]
"""
batch_loop_with_progress = """    for offset in range(0, len(pending_rows), batch_size):
        batch = pending_rows[offset : offset + batch_size]
        print(
            f"Downloading selected light curves "
            f"{min(offset + len(batch), len(pending_rows))}/{len(pending_rows)}",
            flush=True,
        )
"""

if batch_loop in fetch_source:
    fetch_source = fetch_source.replace(batch_loop, batch_loop_with_progress, 1)
elif "Downloading selected light curves" not in fetch_source:
    raise RuntimeError("Could not add download progress messages to fetch.py")

fetch_file.write_text(fetch_source, encoding="utf-8")

labels_file = PROJECT_ROOT / "src" / "cosmic_black_swan" / "catalog_labels.py"
labels_source = labels_file.read_text(encoding="utf-8")
old_aliases = '("best_class_score", "bestclassscore", "class_score", "score")'
new_aliases = '("best_class_score", "bestclassscore", "class_score", "score", "ClassSc")'

if old_aliases in labels_source:
    labels_source = labels_source.replace(old_aliases, new_aliases, 1)
elif new_aliases not in labels_source:
    raise RuntimeError("Could not find the Gaia score-column mapping in catalog_labels.py")

labels_file.write_text(labels_source, encoding="utf-8")

print("MAST pagination/progress and Gaia ClassSc mapping are ready.")

In [ ]:
import json
import pandas as pd
import subprocess

stage_status = {}

def run_stage(name, args, required=False):
    command = [str(x) for x in args]
    print(f"\n--- {name} ---", flush=True)
    print("$", " ".join(command), flush=True)
    result = subprocess.run(command, check=False)
    stage_status[name] = "OK" if result.returncode == 0 else f"FAILED ({result.returncode})"

    if required and result.returncode != 0:
        raise RuntimeError(f"Required stage failed: {name}")

# Main data run
run_stage(
    "Fetch TESS light curves",
    ["cbs", "fetch", "--data-root", DATA_ROOT,
     "--sectors", "1-10", "--max-lightcurves", "5000",
     "--per-sector-cap", "500", "--batch-size", "5"],
    required=True,
)
run_stage(
    "Extract features",
    ["cbs", "featurize", "--data-root", DATA_ROOT, "--append"],
    required=True,
)
run_stage(
    "Rank unusual light curves",
    ["cbs", "score", "--data-root", DATA_ROOT, "--contamination", "0.02"],
    required=True,
)

# Additional analyses. A catalog or service error is recorded, but won't erase the main results.
run_stage(
    "Cross-match catalogs",
    ["cbs", "catalog-labels", "--data-root", DATA_ROOT, "--match-radius-arcsec", "2"],
)
run_stage(
    "Field-relative screening",
    ["cbs", "conformal-screen", "--data-root", DATA_ROOT],
)
run_stage(
    "Injection-recovery benchmark",
    ["cbs", "injection-recovery", "--data-root", DATA_ROOT,
     "--sample-size", "2000", "--amplitudes-sigma", "1,2,4,8"],
)

# Train the Transformer only if Kaggle has a GPU enabled.
try:
    import torch
    if torch.cuda.is_available():
        run_stage(
            "Train masked Transformer",
            ["cbs", "train-transformer", "--data-root", DATA_ROOT,
             "--epochs", "20", "--batch-size", "64"],
        )
    else:
        stage_status["Train masked Transformer"] = "SKIPPED: enable a Kaggle GPU to run it"
except ImportError:
    stage_status["Train masked Transformer"] = "SKIPPED: PyTorch is unavailable"

# Write a compact report for tomorrow.
manifest_path = DATA_ROOT / "manifest.csv"
features_path = DATA_ROOT / "features.csv"
labels_path = DATA_ROOT / "labels.csv"

manifest = pd.read_csv(manifest_path) if manifest_path.exists() else pd.DataFrame()
features = pd.read_csv(features_path) if features_path.exists() else pd.DataFrame()
labels = pd.read_csv(labels_path) if labels_path.exists() else pd.DataFrame()

summary = {
    "stages": stage_status,
    "manifest_status_counts": (
        {str(k): int(v) for k, v in manifest["status"].value_counts().items()}
        if "status" in manifest else {}
    ),
    "feature_rows": int(len(features)),
    "unique_tic_sources": (
        int(features["source_id"].astype(str)
            .loc[features["source_id"].astype(str).str.fullmatch(r"\d{8,16}")]
            .nunique())
        if "source_id" in features else 0
    ),
    "label_evidence_counts": (
        {str(k): int(v) for k, v in labels["evidence_level"].value_counts().items()}
        if "evidence_level" in labels else {}
    ),
    "score_files": (
        [str(p.relative_to(DATA_ROOT)) for p in (DATA_ROOT / "scores").rglob("*") if p.is_file()]
        if (DATA_ROOT / "scores").exists() else []
    ),
}

summary_path = DATA_ROOT / "overnight_run_summary.json"
summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")
print("\nRun summary saved to:", summary_path)
print(json.dumps(summary, indent=2))